# Wikidata Author Name String Enrichment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MattArtzAnthro/wikidata-tools/blob/main/notebooks/Wikidata_Author_Name_String_Enrichment.ipynb)

Created by [Matt Artz](https://www.mattartz.me/) | [GitHub](https://github.com/MattArtzAnthro) | [ORCID](https://orcid.org/0000-0002-3822-1429)

---

## What This Notebook Does

This notebook enriches existing author name string (P2093) statements in Wikidata with additional metadata from CrossRef. Rather than converting P2093 to P50 (which requires a linked person item), this notebook adds qualifiers and references to the existing P2093 statements to make them more informative and verifiable.

For each author name string, the notebook adds:
- **Series ordinal (P1545)**: Author's position in the byline (1st author, 2nd author, etc.)
- **Object named as (P1932)**: The exact name as it appeared in the publication
- **Affiliation string (P6424)**: The author's institutional affiliation at time of publication
- **Reference**: Stated in Crossref with the CrossRef API URL as reference URL

## Key Features

- **CrossRef Integration**: Fetches author affiliations directly from CrossRef API
- **Exact String Matching**: Uses fuzzy matching to align CrossRef author names with Wikidata P2093 values
- **Batch Processing**: Processes multiple articles efficiently with rate limiting
- **Reference Addition**: Documents data provenance with Crossref references
- **Scholarly Endpoint Aware**: Uses query-scholarly.wikidata.org for article lookups

## Workflow

1. **Upload**: CrossRef-sourced article CSV with DOI column
2. **Lookup**: Find each article in Wikidata's scholarly endpoint to get existing P2093 statements
3. **Fetch**: Get author metadata from CrossRef (affiliation, position)
4. **Match**: Align CrossRef authors to Wikidata P2093 values
5. **Generate**: Create QuickStatements to add qualifiers and references
6. **Export**: Download batch file for upload to QuickStatements

## Important Notes

- This notebook **adds qualifiers to existing P2093 statements** — it does not create new ones
- If a P2093 statement already has these qualifiers, QuickStatements will add duplicates — review before running
- The notebook uses exact string matching against Wikidata to ensure correct statement targeting

## Citation

> Artz, M. (2026). Wikidata Tools. GitHub. https://github.com/MattArtzAnthro/wikidata-tools

*A citable DOI will be available via Zenodo.*

## License

[CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/)

## Setup and Installation

*Install required Python packages and import necessary libraries.*

In [ ]:
!pip install requests pandas ipywidgets rapidfuzz -q

import requests
import pandas as pd
import re
import time
import json
from datetime import datetime
from collections import defaultdict
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets
from io import BytesIO
from rapidfuzz import fuzz

print("Setup complete.")

## Configuration

*Define endpoints, constants, and styling.*

In [ ]:
# Wikidata endpoints
# Since May 2025, scholarly articles are ONLY on the scholarly endpoint
SCHOLARLY_ENDPOINT = "https://query-scholarly.wikidata.org/sparql"
MAIN_ENDPOINT = "https://query.wikidata.org/sparql"

# CrossRef API
CROSSREF_WORKS_URL = "https://api.crossref.org/works/"

# Wikidata Q-IDs for references
CROSSREF_QID = "Q5188229"  # Crossref organization

# Wikidata Property IDs
P2093 = "P2093"  # author name string
P1545 = "P1545"  # series ordinal
P1932 = "P1932"  # object named as
P6424 = "P6424"  # affiliation string
P248 = "S248"    # stated in (as source qualifier)
P854 = "S854"    # reference URL (as source qualifier)

# User agent for API requests
USER_AGENT = "WikidataP2093Enrichment/1.0 (matt@mattartz.me; Wikidata bot)"

# Color palette
COLORS = {
    'bg_primary': '#E7ECEF',
    'text_primary': '#274C77',
    'interactive': '#6096BA',
    'bg_secondary': '#A3CEF1',
    'neutral': '#8B8C89',
    'success': '#28a745',
    'warning': '#ffc107'
}

# Styling for containers
CONTAINER_STYLE = f"""
    background-color: {COLORS['bg_primary']};
    border-left: 5px solid {COLORS['text_primary']};
    border-radius: 10px;
    padding: 15px;
    margin: 10px 0;
"""

print(f"Scholarly endpoint: {SCHOLARLY_ENDPOINT}")
print(f"CrossRef reference QID: {CROSSREF_QID}")

## Helper Functions: SPARQL Queries

*Functions to query Wikidata scholarly endpoint for article P2093 data.*

In [ ]:
def sparql_query(endpoint, query, timeout=30):
    """Execute a SPARQL query and return results."""
    try:
        response = requests.get(
            endpoint,
            params={"query": query, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=timeout
        )
        response.raise_for_status()
        return response.json().get("results", {}).get("bindings", [])
    except requests.exceptions.Timeout:
        print(f"   Timeout querying {endpoint}")
        return []
    except requests.exceptions.RequestException as e:
        print(f"   Request error: {e}")
        return []
    except Exception as e:
        print(f"   Unexpected error: {e}")
        return []


def clean_doi(doi_input):
    """Extract clean DOI from various formats."""
    if not doi_input or not isinstance(doi_input, str):
        return None
    doi_input = str(doi_input).strip()
    for pattern in [r'https?://(?:dx\.)?doi\.org/(.+)', r'doi:(.+)']:
        match = re.search(pattern, doi_input, re.IGNORECASE)
        if match:
            doi_input = match.group(1)
            break
    if re.match(r'^10\.\d+/.+', doi_input):
        return doi_input.strip()
    return None


def lookup_article_p2093(doi):
    """
    Look up an article in the scholarly endpoint by DOI.
    Returns article QID and all P2093 author name strings with their existing qualifiers.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None

    doi_upper = doi_clean.upper()

    # Query for article and P2093 statements with any existing qualifiers
    query = f"""
    SELECT ?article ?articleLabel ?authorNameString ?ordinal ?namedAs ?affiliation WHERE {{
      ?article wdt:P356 "{doi_upper}" .
      OPTIONAL {{
        ?article p:P2093 ?authorStmt .
        ?authorStmt ps:P2093 ?authorNameString .
        OPTIONAL {{ ?authorStmt pq:P1545 ?ordinal }}
        OPTIONAL {{ ?authorStmt pq:P1932 ?namedAs }}
        OPTIONAL {{ ?authorStmt pq:P6424 ?affiliation }}
      }}
      SERVICE wikibase:label {{ bd:serviceParam wikibase:language "en". }}
    }}
    """

    results = sparql_query(SCHOLARLY_ENDPOINT, query)

    if not results:
        return None

    article_qid = results[0].get('article', {}).get('value', '').split('/')[-1]
    article_label = results[0].get('articleLabel', {}).get('value', '')

    # Collect all P2093 author name strings with existing qualifiers
    author_strings = {}
    for r in results:
        name = r.get('authorNameString', {}).get('value', '')
        if name:
            author_strings[name] = {
                'exact_name': name,
                'has_ordinal': bool(r.get('ordinal', {}).get('value', '')),
                'has_named_as': bool(r.get('namedAs', {}).get('value', '')),
                'has_affiliation': bool(r.get('affiliation', {}).get('value', '')),
                'existing_ordinal': r.get('ordinal', {}).get('value', ''),
                'existing_affiliation': r.get('affiliation', {}).get('value', '')
            }

    return {
        'qid': article_qid,
        'label': article_label,
        'doi': doi_clean,
        'author_strings': author_strings
    }


print("SPARQL query functions loaded.")

## Helper Functions: CrossRef API

*Functions to fetch author metadata from CrossRef including affiliations.*

In [ ]:
def get_crossref_author_data(doi):
    """
    Fetch author data from CrossRef API for a given DOI.
    Returns list of authors with: name, affiliation, ordinal.
    """
    doi_clean = clean_doi(doi)
    if not doi_clean:
        return None

    url = f"{CROSSREF_WORKS_URL}{doi_clean}"

    try:
        response = requests.get(
            url,
            headers={"User-Agent": USER_AGENT},
            timeout=30
        )
        response.raise_for_status()
        data = response.json()
    except Exception as e:
        return None

    work = data.get('message', {})
    authors_raw = work.get('author', [])

    authors = []
    for idx, author in enumerate(authors_raw, 1):
        # Build full name
        given = author.get('given', '')
        family = author.get('family', '')
        full_name = f"{given} {family}".strip() if given else family

        # Extract affiliation(s)
        affiliations = []
        for aff in author.get('affiliation', []):
            if aff.get('name'):
                affiliations.append(aff['name'])
        affiliation_str = '; '.join(affiliations) if affiliations else None

        authors.append({
            'name': full_name,
            'given': given,
            'family': family,
            'affiliation': affiliation_str,
            'ordinal': str(idx)
        })

    return {
        'doi': doi_clean,
        'crossref_url': f"https://api.crossref.org/v1/works/{doi_clean}",
        'authors': authors
    }


print("CrossRef API functions loaded.")

## Helper Functions: Author Matching

*Functions to match CrossRef authors to Wikidata P2093 values.*

In [ ]:
def match_crossref_to_p2093(crossref_authors, wikidata_p2093):
    """
    Match CrossRef authors to Wikidata P2093 values using fuzzy matching.
    Returns list of matches with enrichment data.
    """
    matches = []
    used_p2093 = set()  # Track which P2093 values have been matched

    for cr_author in crossref_authors:
        cr_name = cr_author['name'].lower().strip()
        best_match = None
        best_score = 0
        best_p2093_name = None

        for p2093_name, p2093_data in wikidata_p2093.items():
            if p2093_name in used_p2093:
                continue

            # Calculate similarity
            score = fuzz.ratio(cr_name, p2093_name.lower())
            # Also try token sort ratio for name order differences
            token_score = fuzz.token_sort_ratio(cr_name, p2093_name.lower())
            score = max(score, token_score)

            if score > best_score:
                best_score = score
                best_match = p2093_data
                best_p2093_name = p2093_name

        # Require at least 75% match
        if best_score >= 75 and best_match:
            used_p2093.add(best_p2093_name)
            matches.append({
                'p2093_exact': best_match['exact_name'],
                'crossref_name': cr_author['name'],
                'ordinal': cr_author['ordinal'],
                'affiliation': cr_author['affiliation'],
                'match_score': best_score,
                'has_existing_ordinal': best_match['has_ordinal'],
                'has_existing_affiliation': best_match['has_affiliation']
            })

    return matches


print("Author matching functions loaded.")

## Test Endpoint Connections

*Verify both Wikidata scholarly endpoint and CrossRef API are accessible.*

In [ ]:
print("Testing connections...")
print()

# Test scholarly endpoint
print("1. Scholarly endpoint (article P2093 lookup):")
test_doi = "10.1111/var.12275"
result = lookup_article_p2093(test_doi)
if result:
    print(f"   ✓ Found: {result['qid']}")
    print(f"   Title: {result['label'][:60]}..." if len(result['label']) > 60 else f"   Title: {result['label']}")
    print(f"   P2093 statements: {len(result['author_strings'])}")
    for name, data in list(result['author_strings'].items())[:3]:
        quals = []
        if data['has_ordinal']: quals.append('ordinal')
        if data['has_affiliation']: quals.append('affiliation')
        qual_str = f" [has: {', '.join(quals)}]" if quals else " [no qualifiers]"
        print(f"      - \"{name}\"{qual_str}")
else:
    print(f"   No result for DOI {test_doi}")

print()

# Test CrossRef API
print("2. CrossRef API (author metadata):")
cr_result = get_crossref_author_data(test_doi)
if cr_result:
    print(f"   ✓ Found {len(cr_result['authors'])} authors")
    for a in cr_result['authors'][:3]:
        aff_str = f" @ {a['affiliation'][:40]}..." if a.get('affiliation') else " [no affiliation]"
        print(f"      - {a['ordinal']}. {a['name']}{aff_str}")
else:
    print(f"   No CrossRef data for {test_doi}")

print()
print("Connection tests complete.")

## Upload Article Data

*Upload a CSV file with article DOIs.*

In [ ]:
articles_df = None
dois_to_process = []

# File upload widget
file_upload = widgets.FileUpload(
    accept='.csv',
    multiple=False,
    description='Upload CSV'
)

upload_output = widgets.Output()

def on_upload(change):
    global articles_df, dois_to_process
    with upload_output:
        clear_output()
        if file_upload.value:
            try:
                file_info = list(file_upload.value.values())[0]
                articles_df = pd.read_csv(BytesIO(file_info['content']))

                # Filter to only articles if Type column exists
                if 'Type' in articles_df.columns:
                    articles_df = articles_df[articles_df['Type'] == 'journal-article'].copy()

                # Extract valid DOIs
                dois_to_process = []
                for _, row in articles_df.iterrows():
                    doi = clean_doi(row.get('DOI', ''))
                    if doi:
                        dois_to_process.append(doi)

                dois_to_process = list(set(dois_to_process))  # Deduplicate

                print(f"Loaded {len(articles_df)} articles")
                print(f"Valid DOIs to process: {len(dois_to_process)}")
                print()

                # Preview
                print("Sample DOIs:")
                for doi in dois_to_process[:5]:
                    print(f"  {doi}")

            except Exception as e:
                print(f"Error loading file: {e}")

file_upload.observe(on_upload, names='value')

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📁 Upload Article Data</h3>
    <p>Upload a CSV file with a DOI column.</p>
    <p><em>All articles with valid DOIs will be processed.</em></p>
</div>
"""))
display(file_upload)
display(upload_output)

## Process Articles

*For each DOI, look up existing P2093 statements in Wikidata and fetch author metadata from CrossRef.*

In [ ]:
# Store processing results
enrichment_data = {}  # doi -> enrichment info

# Configuration widgets
skip_existing = widgets.Checkbox(
    value=True,
    description='Skip P2093 statements that already have affiliation',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

process_button = widgets.Button(
    description='Process Articles',
    button_style='primary',
    icon='cogs'
)

progress_bar = widgets.IntProgress(
    value=0,
    min=0,
    max=100,
    description='Progress:',
    bar_style='info'
)

process_output = widgets.Output()

def run_processing(button):
    global enrichment_data
    enrichment_data = {}

    with process_output:
        clear_output()

        if not dois_to_process:
            print("Please upload article data first.")
            return

        total = len(dois_to_process)
        progress_bar.max = total
        progress_bar.value = 0

        print(f"Processing {total} articles...")
        print("=" * 60)
        print()

        articles_found = 0
        articles_not_found = 0
        crossref_errors = 0
        authors_to_enrich = 0
        authors_skipped = 0

        for idx, doi in enumerate(dois_to_process):
            progress_bar.value = idx + 1

            # Look up article in Wikidata
            wikidata_info = lookup_article_p2093(doi)
            time.sleep(0.3)

            if not wikidata_info:
                articles_not_found += 1
                continue

            if not wikidata_info['author_strings']:
                # Article exists but has no P2093 statements
                continue

            # Get CrossRef author data
            crossref_info = get_crossref_author_data(doi)
            time.sleep(0.3)

            if not crossref_info:
                crossref_errors += 1
                continue

            # Match CrossRef authors to P2093 values
            matches = match_crossref_to_p2093(
                crossref_info['authors'],
                wikidata_info['author_strings']
            )

            if not matches:
                continue

            # Filter matches based on settings
            filtered_matches = []
            for m in matches:
                # Skip if already has affiliation and user wants to skip those
                if skip_existing.value and m['has_existing_affiliation']:
                    authors_skipped += 1
                    continue
                filtered_matches.append(m)

            if filtered_matches:
                enrichment_data[doi] = {
                    'article_qid': wikidata_info['qid'],
                    'article_title': wikidata_info['label'],
                    'crossref_url': crossref_info['crossref_url'],
                    'matches': filtered_matches
                }
                articles_found += 1
                authors_to_enrich += len(filtered_matches)

        print()
        print("=" * 60)
        print("PROCESSING SUMMARY")
        print("=" * 60)
        print(f"Articles with enrichable P2093: {articles_found}")
        print(f"P2093 statements to enrich: {authors_to_enrich}")
        print(f"P2093 statements skipped (already have data): {authors_skipped}")
        print(f"Articles not in Wikidata: {articles_not_found}")
        print(f"CrossRef lookup errors: {crossref_errors}")
        print()

        # Preview some matches
        if enrichment_data:
            print("Sample enrichments:")
            for doi, data in list(enrichment_data.items())[:3]:
                print(f"  {doi} ({data['article_qid']})")
                for m in data['matches'][:2]:
                    aff_preview = f" @ {m['affiliation'][:30]}..." if m.get('affiliation') else " [no affiliation]"
                    print(f"    → {m['ordinal']}. \"{m['p2093_exact']}\"{aff_preview}")

process_button.on_click(run_processing)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">⚙️ Process Articles</h3>
    <p>Look up P2093 statements in Wikidata and fetch CrossRef metadata.</p>
</div>
"""))
display(widgets.VBox([
    skip_existing,
    process_button,
    progress_bar,
    process_output
]))

## Review Enrichments

*Review the P2093 statements that will be enriched.*

In [ ]:
review_output = widgets.Output()

def show_review():
    with review_output:
        clear_output()

        if not enrichment_data:
            print("No enrichments to review. Run processing first.")
            return

        total_enrichments = sum(len(data['matches']) for data in enrichment_data.values())

        print(f"REVIEW: {total_enrichments} P2093 statements to enrich across {len(enrichment_data)} articles")
        print("=" * 70)
        print()

        for doi, data in enrichment_data.items():
            print(f"📄 {data['article_title'][:60]}..." if len(data['article_title']) > 60 else f"📄 {data['article_title']}")
            print(f"   DOI: {doi}")
            print(f"   Article QID: {data['article_qid']}")
            print()

            for m in data['matches']:
                print(f"   👤 P2093: \"{m['p2093_exact']}\"")
                print(f"      + Series ordinal: {m['ordinal']}")
                print(f"      + Object named as: {m['crossref_name']}")
                if m.get('affiliation'):
                    aff_display = m['affiliation'][:60] + '...' if len(m['affiliation']) > 60 else m['affiliation']
                    print(f"      + Affiliation: {aff_display}")
                else:
                    print(f"      + Affiliation: [none in CrossRef]")
                print(f"      + Reference: Crossref ({data['crossref_url'][:50]}...)")
                print()

            print("-" * 70)
            print()

review_button = widgets.Button(
    description='Review Enrichments',
    button_style='info',
    icon='eye'
)
review_button.on_click(lambda b: show_review())

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">👀 Review Enrichments</h3>
    <p>Review the qualifiers and references that will be added to P2093 statements.</p>
</div>
"""))
display(review_button)
display(review_output)

## Generate QuickStatements

*Generate QuickStatements to add qualifiers and references to existing P2093 statements:*
- Series ordinal (P1545)
- Object named as (P1932)
- Affiliation string (P6424)
- Reference: stated in Crossref with reference URL

In [ ]:
def escape_qs_string(s):
    """Escape a string for QuickStatements V1 format."""
    if not s:
        return s
    # Escape double quotes by doubling them
    return s.replace('"', '""')


def generate_quickstatements(include_ordinal=True, include_named_as=True, include_affiliation=True, include_reference=True):
    """
    Generate QuickStatements V1 format to add qualifiers to existing P2093 statements.

    Format: ARTICLE|P2093|"Author Name"|qualifier1|value1|qualifier2|value2|...|S248|Q5188229|S854|"url"

    QuickStatements will match the existing P2093 statement by value and add the qualifiers.
    """
    qs_lines = []

    for doi, data in enrichment_data.items():
        article_qid = data['article_qid']
        crossref_url = data['crossref_url']

        for m in data['matches']:
            # Start with the P2093 statement that we're adding qualifiers to
            p2093_escaped = escape_qs_string(m['p2093_exact'])
            parts = [article_qid, P2093, f'"{p2093_escaped}"']

            # Add series ordinal (P1545)
            if include_ordinal and m.get('ordinal'):
                parts.extend([P1545, f'"{m["ordinal"]}"'])

            # Add object named as (P1932) - the CrossRef version of the name
            if include_named_as and m.get('crossref_name'):
                name_escaped = escape_qs_string(m['crossref_name'])
                parts.extend([P1932, f'"{name_escaped}"'])

            # Add affiliation string (P6424)
            if include_affiliation and m.get('affiliation'):
                aff_escaped = escape_qs_string(m['affiliation'])
                parts.extend([P6424, f'"{aff_escaped}"'])

            # Add references: stated in Crossref (S248), reference URL (S854)
            if include_reference:
                parts.extend([P248, CROSSREF_QID])
                url_escaped = escape_qs_string(crossref_url)
                parts.extend([P854, f'"{url_escaped}"'])

            qs_lines.append('|'.join(parts))

    return '\n'.join(qs_lines)


# Option checkboxes
include_ordinal_cb = widgets.Checkbox(
    value=True,
    description='Include series ordinal (P1545)',
    style={'description_width': 'initial'}
)

include_named_as_cb = widgets.Checkbox(
    value=True,
    description='Include object named as (P1932)',
    style={'description_width': 'initial'}
)

include_affiliation_cb = widgets.Checkbox(
    value=True,
    description='Include affiliation string (P6424)',
    style={'description_width': 'initial'}
)

include_reference_cb = widgets.Checkbox(
    value=True,
    description='Include Crossref reference',
    style={'description_width': 'initial'}
)

generate_button = widgets.Button(
    description='Generate QuickStatements',
    button_style='success',
    icon='download'
)

generate_output = widgets.Output()

def run_generate(button):
    with generate_output:
        clear_output()

        if not enrichment_data:
            print("No enrichments to generate. Run processing first.")
            return

        print("Generating QuickStatements...")
        print()

        qs_text = generate_quickstatements(
            include_ordinal=include_ordinal_cb.value,
            include_named_as=include_named_as_cb.value,
            include_affiliation=include_affiliation_cb.value,
            include_reference=include_reference_cb.value
        )

        if not qs_text:
            print("No QuickStatements generated.")
            return

        # Save to file
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"p2093_enrichment_{timestamp}.txt"

        with open(filename, 'w', encoding='utf-8') as f:
            f.write(qs_text)

        line_count = len(qs_text.strip().split('\n'))
        total_enrichments = sum(len(data['matches']) for data in enrichment_data.values())

        print(f"Generated {line_count} QuickStatements commands")
        print(f"Enriching {total_enrichments} P2093 statements across {len(enrichment_data)} articles")
        print(f"Saved to: {filename}")
        print()

        # Preview
        print("--- PREVIEW (first 3 commands) ---")
        preview_lines = qs_text.strip().split('\n')[:3]
        for line in preview_lines:
            # Truncate long lines for display
            if len(line) > 100:
                print(line[:100] + "...")
            else:
                print(line)
        print()

        print("--- FORMAT EXPLANATION ---")
        print('Format: ARTICLE|P2093|"Name"|P1545|"ord"|P1932|"name"|P6424|"affil"|S248|Q5188229|S854|"url"')
        print()
        print("Qualifiers added to existing P2093:")
        if include_ordinal_cb.value:
            print("  - P1545: Series ordinal (author position)")
        if include_named_as_cb.value:
            print("  - P1932: Object named as (name from CrossRef)")
        if include_affiliation_cb.value:
            print("  - P6424: Affiliation string")
        if include_reference_cb.value:
            print("  - S248: Stated in Crossref (Q5188229)")
            print("  - S854: Reference URL (CrossRef API)")
        print()

        print("--- UPLOAD INSTRUCTIONS ---")
        print("1. Go to: https://quickstatements.toolforge.org/")
        print("2. Log in with your Wikidata account")
        print("3. Click 'New batch'")
        print("4. Paste the file contents")
        print("5. Click 'Import V1 commands'")
        print("6. Review carefully and click 'Run'")
        print()
        print("NOTE: These commands ADD qualifiers to existing P2093 statements.")
        print("QuickStatements matches by the P2093 string value.")

        # Download in Colab
        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

generate_button.on_click(run_generate)

# Display
display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📤 Generate QuickStatements</h3>
    <p>Generate QuickStatements to add qualifiers and references to existing P2093 statements.</p>
    <p><strong>Select which data to include:</strong></p>
</div>
"""))
display(widgets.VBox([
    include_ordinal_cb,
    include_named_as_cb,
    include_affiliation_cb,
    include_reference_cb,
    generate_button,
    generate_output
]))

## Export Summary

*Export a CSV summary of all P2093 enrichments for documentation.*

In [ ]:
export_button = widgets.Button(
    description='Export Summary CSV',
    button_style='info',
    icon='table'
)

export_output = widgets.Output()

def run_export(button):
    with export_output:
        clear_output()

        if not enrichment_data:
            print("No data to export. Run processing first.")
            return

        rows = []
        for doi, data in enrichment_data.items():
            for m in data['matches']:
                rows.append({
                    'DOI': doi,
                    'Article_QID': data['article_qid'],
                    'Article_Title': data['article_title'],
                    'P2093_Value': m['p2093_exact'],
                    'CrossRef_Name': m['crossref_name'],
                    'Ordinal': m['ordinal'],
                    'Affiliation': m.get('affiliation', ''),
                    'Match_Score': m['match_score'],
                    'CrossRef_URL': data['crossref_url']
                })

        df = pd.DataFrame(rows)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"p2093_enrichment_summary_{timestamp}.csv"

        df.to_csv(filename, index=False)

        # Stats
        with_affiliation = len([r for r in rows if r['Affiliation']])

        print(f"Exported: {filename}")
        print(f"Total P2093 enrichments: {len(rows)}")
        print(f"Unique articles: {len(enrichment_data)}")
        print(f"With affiliation data: {with_affiliation} ({100*with_affiliation/len(rows):.1f}%)")

        try:
            from google.colab import files
            files.download(filename)
        except:
            pass

export_button.on_click(run_export)

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📊 Export Summary</h3>
    <p>Export a CSV summary of all P2093 enrichments for documentation.</p>
</div>
"""))
display(export_button)
display(export_output)

## Statistics

*View statistics on affiliation coverage and enrichment rates.*

In [ ]:
stats_button = widgets.Button(
    description='Show Statistics',
    button_style='info',
    icon='bar-chart'
)

stats_output = widgets.Output()

def show_stats(button):
    with stats_output:
        clear_output()

        if not enrichment_data:
            print("No data for statistics. Run processing first.")
            return

        print("P2093 ENRICHMENT STATISTICS")
        print("=" * 50)
        print()

        total_enrichments = sum(len(data['matches']) for data in enrichment_data.values())

        print(f"Overview:")
        print(f"  Articles to enrich: {len(enrichment_data)}")
        print(f"  P2093 statements to enrich: {total_enrichments}")
        print()

        # Affiliation coverage
        with_affiliation = 0
        without_affiliation = 0

        for data in enrichment_data.values():
            for m in data['matches']:
                if m.get('affiliation'):
                    with_affiliation += 1
                else:
                    without_affiliation += 1

        print(f"Affiliation Coverage:")
        print(f"  With affiliation: {with_affiliation} ({100*with_affiliation/total_enrichments:.1f}%)")
        print(f"  Without affiliation: {without_affiliation} ({100*without_affiliation/total_enrichments:.1f}%)")
        print()

        # Match quality
        scores = [m['match_score'] for data in enrichment_data.values() for m in data['matches']]
        avg_score = sum(scores) / len(scores)
        high_confidence = len([s for s in scores if s >= 90])
        medium_confidence = len([s for s in scores if 75 <= s < 90])

        print(f"Name Match Quality:")
        print(f"  Average match score: {avg_score:.1f}%")
        print(f"  High confidence (≥90%): {high_confidence}")
        print(f"  Medium confidence (75-89%): {medium_confidence}")

stats_button.on_click(show_stats)

display(HTML(f"""
<div style="{CONTAINER_STYLE}">
    <h3 style="color: {COLORS['text_primary']}; margin-top: 0;">📈 Statistics</h3>
    <p>View enrichment and affiliation coverage statistics.</p>
</div>
"""))
display(stats_button)
display(stats_output)